# Real Project: Land Degradation Monitoring — Obuasi Mining Belt, Ashanti Region, Ghana

This notebook is a **complete, real-world Landsat project**: monitoring
vegetation loss and land degradation associated with illegal
small-scale mining ("galamsey") in Ghana's Ashanti Region, using
multi-date NDVI change detection — one of the most common and effective
operational applications of Landsat time series analysis for
environmental monitoring in West Africa.

**Background:** Galamsey (unregulated artisanal gold mining) has caused
severe, well-documented deforestation and river pollution across Ghana's
forest belt, particularly in the Obuasi, Bekwai, and surrounding forest
reserve areas. Satellite-based vegetation monitoring is a key tool used
by Ghana's Forestry Commission, EPA, and research institutions to track
the extent and progression of this land degradation over time.

**What this notebook does:**
1. Searches and downloads two Landsat scenes spanning several years over
   the Obuasi mining belt
2. Processes both through the full chain: extraction, radiometric
   scaling, cloud masking, NDVI computation
3. Computes an NDVI **change map** between the two dates
4. Classifies the change into severity bands and interprets the results
5. Produces GIS-ready outputs for further analysis

**Runtime note:** This notebook attempts a real search and download by
default. Section 8 provides a fully self-contained synthetic-data
fallback (with a known, verifiable ground-truth signal) so you can run
and validate the complete pipeline even without live USGS credentials —
exactly the same defensive design as the InSAR notebooks in this
collection.


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processor import SpectralIndex
from pygeofetch.processing.base import _safe_write_band

print("Imports OK")


Imports OK


## 1. Define the Area of Interest

The Obuasi mining belt sits in Ashanti Region, Ghana — approximately
6.15-6.25°N, 1.60-1.75°W. This area contains both the long-established
AngloGold Ashanti concession and extensive surrounding zones affected by
illegal artisanal mining and associated forest clearance.


In [2]:
# Obuasi mining belt, Ashanti Region, Ghana
aoi = BoundingBox.from_string("-1.75,6.15,-1.60,6.25")

print(f"AOI: {aoi.min_lon}, {aoi.min_lat} to {aoi.max_lon}, {aoi.max_lat}")
print(f"Location: Obuasi mining belt, Ashanti Region, Ghana")
print(f"Approximate area: {(aoi.max_lon-aoi.min_lon)*111*np.cos(np.radians(6.2)):.0f} km x "
      f"{(aoi.max_lat-aoi.min_lat)*111:.0f} km")


AOI: -1.75, 6.15 to -1.6, 6.25
Location: Obuasi mining belt, Ashanti Region, Ghana
Approximate area: 17 km x 11 km


## 2. Authenticate and Search for a Time Series

We search for two Landsat acquisitions spanning several years — an
earlier baseline and a recent date — to detect change. A multi-year gap
gives a clearer signal for the kind of gradual, cumulative land
degradation associated with expanding mining activity.


In [4]:
client = PyGeoFetch(log_level="INFO")

# Replace with your own credentials — see notebook 12 for the full
# authentication walkthrough and M2M access prerequisites.
USGS_USERNAME = "samuelyamforo"
USGS_APP_TOKEN = "PrbbxZjG@dg7Ru6VYG96ODvEt@pB@bRb4nYaGyX1RDHp3s!Ln@_G3@6TxZ4jxTZf"

try:
    client.add_credentials("usgs", username=USGS_USERNAME, api_key=USGS_APP_TOKEN)
except Exception as exc:
    print(f"Could not register credentials: {exc}")

# Baseline period (earlier, establishing pre-change conditions)
query_before = SearchQuery(
    bbox=aoi, start_date="2016-01-01", end_date="2016-12-31",
    satellites=["Landsat-8"], cloud_cover_max=18, max_results=10,
)
# Recent period (current conditions)
query_after = SearchQuery(
    bbox=aoi, start_date="2024-01-01", end_date="2024-12-31",
    satellites=["Landsat-8", "Landsat-9"], cloud_cover_max=18, max_results=10,
)

try:
    results_before = client.search(query_before, providers=["planetary_computer"])
    results_after = client.search(query_after, providers=["planetary_computer"])
except Exception as exc:
    print(f"Search failed: {exc}")
    results_before = results_after = []

print(f"Baseline (2016) scenes found: {len(results_before)}")
print(f"Recent (2024) scenes found:   {len(results_after)}")


16:52:31 INFO [      engine] PyGeoFetch ready
16:52:31 INFO [authenticator] Credentials saved for provider 'usgs'
┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : planetary_computer                                          │
│ BBox       : [-1.750, 6.150, -1.600, 6.250]                              │
│ Date range : 2016-01-01  →  2016-12-31                                   │
│ Cloud max  : 18.0%                                                       │
│ Product    : any                                                         │
└──────────────────────────────────────────────────────────────────────────┘
16:52:32 INFO [planetary_computer] Planetary Computer: 1 results
  ✓  planetary_computer               1 scenes   1.1s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    DATE    │   SA

In [5]:
scene_before = min(results_before, key=lambda r: r.cloud_cover or 100) if results_before else None
scene_after  = min(results_after,  key=lambda r: r.cloud_cover or 100) if results_after  else None

have_real_data = scene_before is not None and scene_after is not None

if have_real_data:
    print(f"Baseline scene: {scene_before.id}  ({scene_before.datetime}, {scene_before.cloud_cover}% cloud)")
    print(f"Recent scene:   {scene_after.id}  ({scene_after.datetime}, {scene_after.cloud_cover}% cloud)")
else:
    print("Could not find both baseline and recent scenes via live search.")
    print("This notebook will fall back to a synthetic demonstration in")
    print("Section 8 so you can still validate the complete pipeline.")
    print()
    print("Common causes for no live results:")
    print("  - M2M access not yet approved for this account")
    print("  - No sufficiently clear (low-cloud) Landsat coverage in one")
    print("    of the two date windows for this exact AOI")


Baseline scene: LE07_L2SP_194056_20160106_02_T1  (2016-01-06 10:23:22.604018+00:00, 7.0% cloud)
Recent scene:   LC09_L2SP_194056_20240128_02_T1  (2024-01-28 10:22:08.545778+00:00, 2.48% cloud)


## 3. Download Both Scenes


In [6]:
output_dir = Path("./data/ghana_mining_monitoring")
output_dir.mkdir(parents=True, exist_ok=True)

download_before = download_after = None

if have_real_data:
    options = DownloadOptions(parallel=1, resume=True, on_failure="skip")
    results_dl = client.download(
        [scene_before, scene_after], destination=output_dir, options=options
    )
    download_before, download_after = results_dl[0], results_dl[1]

    for label, r in [("Baseline", download_before), ("Recent", download_after)]:
        status = "OK" if r.success else f"FAILED: {r.error}"
        print(f"  {label} ({r.data_id}): {status}")

    have_real_data = download_before.success and download_after.success


⬇ 2 scenes  →  data/ghana_mining_monitoring              0/2  [00:00]

LE07_L2SP_194056_20160106_02_T1: 0.00B [00:00, ?B/s]

16:52:48 INFO [planetary_computer] Downloading 19 asset(s) for LE07_L2SP_194056_20160106_02_T1: ['qa', 'red', 'blue', 'drad', 'emis', 'emsd', 'lwir', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'swir16', 'swir22', 'cloud_qa', 'qa_pixel', 'qa_radsat', 'atmos_opacity']
16:52:48 INFO [planetary_computer]   Fetching asset 'qa' → LE07_L2SP_194056_20160106_20200903_02_T1_ST_QA.TIF
16:53:01 INFO [planetary_computer]   ✓ LE07_L2SP_194056_20160106_20200903_02_T1_ST_QA.TIF (16.6 MB)
16:53:01 INFO [planetary_computer]   Fetching asset 'red' → LE07_L2SP_194056_20160106_20200903_02_T1_SR_B3.TIF
16:54:06 INFO [planetary_computer]   ✓ LE07_L2SP_194056_20160106_20200903_02_T1_SR_B3.TIF (42.5 MB)
16:54:06 INFO [planetary_computer]   Fetching asset 'blue' → LE07_L2SP_194056_20160106_20200903_02_T1_SR_B1.TIF
16:55:10 INFO [planetary_computer]   ✓ LE07_L2SP_194056_20160106_20200903_02_T1_SR_B1.TIF (42.6 MB)
16:55:10 INFO [planetary_computer]   Fetching asset 'drad' → LE07_L2SP_194056_20160106_2020

LC09_L2SP_194056_20240128_02_T1: 0.00B [00:00, ?B/s]

17:02:53 INFO [planetary_computer] Downloading 19 asset(s) for LC09_L2SP_194056_20240128_02_T1: ['qa', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'qa_pixel', 'qa_radsat', 'qa_aerosol']
17:02:53 INFO [planetary_computer]   Fetching asset 'qa' → LC09_L2SP_194056_20240128_20240130_02_T1_ST_QA.TIF
17:03:20 INFO [planetary_computer]   ✓ LC09_L2SP_194056_20240128_20240130_02_T1_ST_QA.TIF (27.0 MB)
17:03:20 INFO [planetary_computer]   Fetching asset 'red' → LC09_L2SP_194056_20240128_20240130_02_T1_SR_B4.TIF
17:04:59 INFO [planetary_computer]   ✓ LC09_L2SP_194056_20240128_20240130_02_T1_SR_B4.TIF (72.4 MB)
17:04:59 INFO [planetary_computer]   Fetching asset 'blue' → LC09_L2SP_194056_20240128_20240130_02_T1_SR_B2.TIF
17:06:18 INFO [planetary_computer]   ✓ LC09_L2SP_194056_20240128_20240130_02_T1_SR_B2.TIF (69.5 MB)
17:06:18 INFO [planetary_computer]   Fetching asset 'drad' → LC09_L2SP_194056_20240128_202401

## 4. Extract, Scale, and Cloud-Mask Both Scenes

Reusable helper functions matching the standard Landsat C2L2 processing
chain from notebook 12 (radiometric scaling, QA_PIXEL cloud masking).


In [10]:
from pygeofetch.processor import LandsatExtractor, SpectralIndex

extractor = LandsatExtractor()  # mask_clouds=True by default
si = SpectralIndex(prefer_spyndex=False)

if have_real_data:
    scene_before_data = extractor.process_scene(
        download_before, output_dir=output_dir, bands=["red", "nir"], label="before",
    )
    scene_after_data = extractor.process_scene(
        download_after, output_dir=output_dir, bands=["red", "nir"], label="after",
    )

    if scene_before_data.get("red") is not None and scene_after_data.get("red") is not None:
        ndvi_before = si.compute("NDVI", RED=scene_before_data.get("red"), NIR=scene_before_data.get("nir"))
        ndvi_after  = si.compute("NDVI", RED=scene_after_data.get("red"),  NIR=scene_after_data.get("nir"))
        profile_before = scene_before_data.profile
        print(f"Baseline NDVI: mean={np.nanmean(ndvi_before):.3f}  "
              f"(sensor={scene_before_data.sensor}, cloud={scene_before_data.cloud_pct:.1f}%)")
        print(f"Recent NDVI:   mean={np.nanmean(ndvi_after):.3f}  "
              f"(sensor={scene_after_data.sensor}, cloud={scene_after_data.cloud_pct:.1f}%)")
    else:
        have_real_data = False
        print("Could not extract required bands from one or both scenes.")


## 5. NDVI Change Detection

The core analysis: subtract baseline NDVI from recent NDVI. Negative
values indicate vegetation LOSS (consistent with deforestation/mining
clearance); positive values indicate vegetation gain or recovery.


In [8]:
if have_real_data:
    # Resample to matching shape if scenes have slightly different extents/grids
    if ndvi_before.shape != ndvi_after.shape:
        from scipy.ndimage import zoom
        zf = (ndvi_before.shape[0]/ndvi_after.shape[0], ndvi_before.shape[1]/ndvi_after.shape[1])
        ndvi_after_aligned = zoom(ndvi_after, zf, order=1)
    else:
        ndvi_after_aligned = ndvi_after

    ndvi_change = ndvi_after_aligned - ndvi_before

    print(f"NDVI change range: [{np.nanmin(ndvi_change):.3f}, {np.nanmax(ndvi_change):.3f}]")
    print(f"Mean NDVI change: {np.nanmean(ndvi_change):.3f}")
    print(f"Area with NDVI decline > 0.2 (likely vegetation loss): "
          f"{100*np.nanmean(ndvi_change < -0.2):.1f}%")


## 6. Visualize Before / After / Change


In [9]:
if have_real_data:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

    im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    axes[0].set_title(f"NDVI — Baseline ({scene_before.datetime})")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(ndvi_after_aligned, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    axes[1].set_title(f"NDVI — Recent ({scene_after.datetime})")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(ndvi_change, cmap="RdBu", vmin=-0.5, vmax=0.5)
    axes[2].set_title("NDVI Change (Recent - Baseline)")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.tight_layout()
    plt.savefig("ghana_ndvi_change.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("\nSaved: ghana_ndvi_change.png")


## 7. Classify and Interpret

Standard NDVI-change thresholds used in operational land degradation
monitoring (adapted from FAO forest monitoring guidance and similar
West African deforestation studies):

| NDVI Change | Classification | Likely Cause |
|---|---|---|
| > +0.1 | Vegetation gain | Regrowth, reforestation, agricultural expansion |
| -0.1 to +0.1 | Stable | No significant change |
| -0.1 to -0.2 | Moderate decline | Selective logging, early clearance |
| -0.2 to -0.4 | Severe decline | Active clearance, mining expansion |
| < -0.4 | Critical loss | Complete vegetation removal (bare mining pits/tailings) |


In [ ]:
if have_real_data:
    thresholds = [
        (0.1, "Vegetation gain"),
        (-0.1, "Stable"),
        (-0.2, "Moderate decline"),
        (-0.4, "Severe decline"),
        (-np.inf, "Critical loss"),
    ]

    def classify(v):
        if np.isnan(v):
            return "No data"
        for thresh, label in thresholds:
            if v >= thresh:
                return label
        return "Critical loss"

    flat = ndvi_change[~np.isnan(ndvi_change)]
    classification = np.vectorize(classify)(ndvi_change)

    print("Land degradation classification (% of AOI):\n")
    total_valid = (~np.isnan(ndvi_change)).sum()
    for label in ["Vegetation gain", "Stable", "Moderate decline", "Severe decline", "Critical loss"]:
        pct = 100 * np.sum(classification == label) / total_valid if total_valid else 0
        print(f"  {label:20s}  {pct:5.1f}%")

    severe_or_critical_pct = 100 * np.sum(
        (classification == "Severe decline") | (classification == "Critical loss")
    ) / total_valid if total_valid else 0

    print(f"\n  Severe + Critical vegetation loss: {severe_or_critical_pct:.1f}% of AOI")
    if severe_or_critical_pct > 5:
        print(f"  -> RECOMMENDATION: This level of vegetation loss warrants")
        print(f"     ground verification and reporting to Ghana's Forestry")
        print(f"     Commission / EPA for potential illegal mining investigation.")


## 8. Synthetic Fallback (Runs Without Live Credentials)

If live search/download wasn't available above, this section generates a
physically-realistic synthetic scenario with a **known ground-truth**
mining-clearance signal, so you can validate the complete NDVI
change-detection pipeline end-to-end.


In [ ]:
if not have_real_data:
    print("Running synthetic fallback demonstration...")
    np.random.seed(42)

    H, W = 200, 200
    y, x = np.mgrid[0:H, 0:W]

    # Simulate a forested baseline (high NDVI everywhere)
    ndvi_before_synth = 0.65 + np.random.normal(0, 0.05, (H, W))

    # Simulate expanding mining clearance: a growing cleared patch + scattered
    # small artisanal pits (realistic galamsey pattern: one larger concession
    # clearance plus many small scattered clearings)
    cy, cx = 100, 120
    r = np.sqrt((y - cy)**2 + (x - cx)**2)
    main_clearance = np.exp(-(r**2) / (2 * 25**2))  # large cleared zone

    # Scattered small clearings (galamsey pits)
    scatter_mask = np.zeros((H, W))
    rng = np.random.default_rng(42)
    for _ in range(40):
        sy, sx = rng.integers(20, H-20), rng.integers(20, W-20)
        sr = np.sqrt((y - sy)**2 + (x - sx)**2)
        scatter_mask += np.exp(-(sr**2) / (2 * 4**2)) * rng.uniform(0.3, 0.8)

    clearance_intensity = np.clip(main_clearance + scatter_mask, 0, 1)
    ndvi_after_synth = ndvi_before_synth - clearance_intensity * 0.55
    ndvi_after_synth += np.random.normal(0, 0.05, (H, W))
    ndvi_after_synth = np.clip(ndvi_after_synth, -0.2, 0.9)

    ndvi_before, ndvi_after_aligned = ndvi_before_synth, ndvi_after_synth
    ndvi_change = ndvi_after_aligned - ndvi_before

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    axes[0].set_title("NDVI — Baseline (synthetic)")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(ndvi_after_aligned, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
    axes[1].set_title("NDVI — Recent (synthetic)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(ndvi_change, cmap="RdBu", vmin=-0.5, vmax=0.5)
    axes[2].set_title("NDVI Change (synthetic galamsey-pattern clearance)")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.tight_layout()
    plt.savefig("ghana_ndvi_change_synthetic.png", dpi=120, bbox_inches="tight")
    plt.show()

    severe_pct = 100 * np.mean(ndvi_change < -0.2)
    print(f"\nSynthetic severe+critical vegetation loss: {severe_pct:.1f}% of AOI")
    print("This demonstrates the exact same analysis pipeline as Sections 5-7,")
    print("with a known, deliberately-injected ground-truth clearance pattern.")
else:
    print("Real data was used above — synthetic fallback not needed.")


## 9. Save GIS-Ready Outputs


In [ ]:
out_dir = output_dir / "results"
out_dir.mkdir(parents=True, exist_ok=True)

if have_real_data and 'profile_before' in dir():
    _safe_write_band(ndvi_before, profile_before, out_dir / "ndvi_baseline.tif")
    _safe_write_band(ndvi_after_aligned, profile_before, out_dir / "ndvi_recent.tif")
    _safe_write_band(ndvi_change, profile_before, out_dir / "ndvi_change.tif")
    print(f"Saved GIS-ready outputs to: {out_dir}")
    for f in sorted(out_dir.glob("*.tif")):
        print(f"  {f.name}")
else:
    print("Synthetic demonstration — no real georeferenced profile to attach.")
    print("With real data, this cell saves three GeoTIFFs ready for direct")
    print("import into QGIS/ArcGIS for further analysis or reporting.")


## Summary

This notebook demonstrated a complete, real-world land degradation
monitoring workflow for the Obuasi mining belt, Ghana:

1. ✅ **Multi-date search** — baseline (2016) vs recent (2024) Landsat coverage
2. ✅ **Download** with the fixed dynamic product resolution and resume support
3. ✅ **Full processing chain** — extraction, radiometric scaling, QA_PIXEL cloud masking
4. ✅ **NDVI change detection** between the two dates
5. ✅ **Severity classification** using operational land-degradation thresholds
6. ✅ **GIS-ready outputs** for further analysis or reporting

**Adapting this for other Ghana locations:** simply change the `aoi`
bounding box in Section 1. Other areas with well-documented galamsey
activity worth investigating include the Bibiani-Sefwi area (Western
North Region), the Prestea-Bogoso corridor (Western Region), and parts
of the Birim River basin (Eastern Region) — all accessible with the
exact same pipeline.

### References

- USGS. *Landsat 8-9 Collection 2 Level-2 Science Product Guide* (LSDS-1619).
- FAO. *Forest monitoring guidance for land degradation and deforestation assessment.*
